In [ ]:
from core.config import ConfigManager
from astropy.coordinates import SkyCoord
from pathlib import Path
import os
from core.logger import PipelineLogger
from core.directory_manager import DirectoryManager
from core.hdf5_handler import HDF5Handler
from core.map_tools import MapGenerator

In [ ]:
config = ConfigManager('config.yaml')
method = config.get('fitting_procedure')
coordsys = config.get('coordinates.coord_sys', 'equatorial')
output_path = config.get('fitting.output_dir')
output_dir_name = config.get('fitting.fit_name')

# Test logger
logger = PipelineLogger('./logs')
logger.info("Pipeline test")

directory_manager = DirectoryManager(output_path, output_dir_name, logger=logger)
directory_manager.create_structure()

ra = config.get('coordinates.ra')
dec = config.get('coordinates.dec')
if ra is None or dec is None:
    l = config.get('coordinates.l')
    b = config.get('coordinates.b')
    if l is None or b is None:
        raise ValueError("Config must set either (ra, dec) or (l, b) for the ROI center")
    c = SkyCoord(l, b, frame='galactic', unit='deg')
    ra = float(c.icrs.ra.deg)
    dec = float(c.icrs.dec.deg)
    logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
    config.set('coordinates.ra', ra)
    config.set('coordinates.dec', dec)


print(f"Fitting method: {method}")
print(f"Coordinate system: {coordsys}")
print(f"RA: {ra}")
print(f"Dec: {dec}")
print(f"Output path: {output_path}")
print(f"Output directory name: {output_dir_name}")

In [ ]:
###Drips 
step_dir = directory_manager.get_step_results_dir('Step0-Allpoint-sources')

In [ ]:
def _build_significance_map():
        """Build the significance map from count maps if coordinates.create_sig_map."""
        if not config.get('coordinates.create_sig_map', False):
            return None
        sig_map_path_cfe = config.get('coordinates.sig_map_path')
        if sig_map_path_cfe:
            sig_map_path = Path(sig_map_path_cfe)
        else:
            sig_map_path = directory_manager.get_datamap_dir() / "significance_map.fits"
            logger.info(f"Significance map already exists at {sig_map_path}, skipping generation")
            config.set("coordinates.sig_map_path", str(sig_map_path))
        if sig_map_path.exists():
            logger.info(f"Significance map already exists at {sig_map_path}, skipping generation")
            return sig_map_path
        
        # checkpoint.save_step('build_significance_map', 0, 'running', {})
        count_map_dir = config.get('coordinates.count_map_dir')
        image_bins = config.get('coordinates.image_bins')
        detector_response = config.get('coordinates.detector_response')

        fits_mapping = MapGenerator.find_fits_files_by_bins(count_map_dir, image_bins, logger=logger)
        if not fits_mapping:
            checkpoint.save_step('build_significance_map', 0, 'failed', {'error': 'no count-map FITS files found'})
            raise RuntimeError(f"No count-map FITS files found in {count_map_dir} for bins {image_bins}")
        ra = config.get('coordinates.ra')
        dec = config.get('coordinates.dec')
        if ra is None or dec is None:
            l = config.get('coordinates.l')
            b = config.get('coordinates.b')
            skycoord = SkyCoord(l, b, frame='galactic', unit='deg')
            ra = skycoord.icrs.ra.deg
            dec = skycoord.icrs.dec.deg
            logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
            config.set('coordinates.ra', ra)
            config.set('coordinates.dec', dec) 
        
        output_path = MapGenerator.create_healpix_map(
            input_fits_files=list(fits_mapping.values()),
            energy_bins=list(fits_mapping.keys()),
            detector_response=detector_response,
            ra_center=float(config.get('coordinates.ra')),
            dec_center=float(config.get('coordinates.dec')),
            roi_x=float(10.0),
            roi_y=float(10.0),
            output_file=str(sig_map_path),
            logger=logger,
            pixi_manifest_path=config.get('alps.pixi_aerie_folder'),
        )


In [ ]:
_build_significance_map()

In [ ]:
%load_ext autoreload
%autoreload 2
from drips_seeder import DRIPSSeeder
config = ConfigManager('config.yaml')
seeder = DRIPSSeeder(config, logger, directory_manager, step_path=str(step_dir))
drip_model_path = seeder.run()

In [ ]:
%load_ext autoreload
%autoreload 2
import importlib
import source_fitter
import astromodels
importlib.reload(source_fitter)
logger.info('Starting source_fitter (DRIPS-seeded in-process fit)')

fit_output = source_fitter.run_joint_fit(drip_model_path, config, logger, directory_manager)

In [ ]:
def _build_fit_maps(config, logger, directory_manager, path, name, checkpoint=None):
    """Create significance maps."""

    ra = config.get('coordinates.ra')
    dec = config.get('coordinates.dec')
    if ra is None or dec is None:
        l = config.get('coordinates.l')
        b = config.get('coordinates.b')
        skycoord = SkyCoord(l, b, frame='galactic', unit='deg')
        ra = skycoord.icrs.ra.deg
        dec = skycoord.icrs.dec.deg
        logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
        config.set('coordinates.ra', ra)
        config.set('coordinates.dec', dec) 
    
    
    created_files = HDF5Handler.convert_hd5_to_fits(
        input_dir=str(path),
        hd5_filename='residual_fit.hd5',
        output_basename='residual',
        logger=logger
    )
    print(f"Created FITS files: {created_files}")
    bins = config.get('fitting.bins')
    detector_response = config.get('coordinates.detector_response')
    print(f"Path: {path}")

    if os.path.exists(str(path / 'fits' / f'{name}.fits')):
        logger.info(f"Output FITS file {path / 'fits' / f'{name}.fits'} already exists, skipping map generation")
        return path / 'fits' / f'{name}.fits' 
    output_path = MapGenerator.create_healpix_map(
        input_fits_files=list(created_files),
        energy_bins=list(bins),
        detector_response=detector_response,
        ra_center=float(config.get('coordinates.ra')),
        dec_center=float(config.get('coordinates.dec')),
        roi_x=float(config.get('coordinates.roi_x', 4.0)*2.5),
        roi_y=float(config.get('coordinates.roi_y', 4.0)*2),
        output_file=str(path / 'fits' / f'{name}.fits'),
        logger=logger,
        pixi_manifest_path=config.get('alps.pixi_aerie_folder'),
    )

    return output_path
# def check_hotspots(residual_map_path):


In [ ]:
resmap = _build_fit_maps(config, logger, directory_manager, fit_output.step_dir, 'residual')

In [ ]:
from pipeline_helpers import load_hawc_data, find_peak, make_plots
def check_hotspots(path, fit_output, config, logger):
    mapname = path
    l = config.get('coordinates.l')
    b = config.get('coordinates.b')
    x_length = config.get('coordinates.roi_x')
    y_length = config.get('coordinates.roi_y')
    coord_sys = config.get('coordinates.coord_sys')
    array, _, wcs, _, _, pixel_size = load_hawc_data( mapname, l, b, x_length, y_length, coord_sys )
    max_value = find_peak(array, wcs)
    print(f"path.parent: {path.parent.parent}")
    if max_value > 5:
        name = []
        ra = []
        dec = []
        ext = []
        for source in fit_output.model.sources:
            logger.info(f"Source {source}")
            if source == 'URM':
                continue
            try:
                logger.info(f"Source position: RA={fit_output.model[source].position.ra.value}, Dec={fit_output.model[source].position.dec.value}")
                name.append(source)
                ra.append(fit_output.model[source].position.ra.value)
                dec.append(fit_output.model[source].position.dec.value)
                ext.append(0.01)
            except:
                logger.info(f"Source position: RA={fit_output.model[source].spatial_shape.lon0.value}, Dec={fit_output.model[source].spatial_shape.lat0.value}")
                name.append(source)
                ra.append(fit_output.model[source].spatial_shape.lon0.value)
                dec.append(fit_output.model[source].spatial_shape.lat0.value)
                ext.append(fit_output.model[source].spatial_shape.sigma.value)
        df = {'Name': name, 'ra': ra, 'dec': dec, 'ext': ext}
        make_plots(array, wcs, pixel_size, coord_sys, save_dir = str(path.parent), cmap='ult', hotspots=df)
    logger.info(f"Max value in residual map: {max_value}")

In [ ]:
check_hotspots(resmap, fit_output, config, logger)

In [ ]:
from typing import List
def _as_list(value) -> List[str]:
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


In [ ]:
config = ConfigManager('config.yaml')
model = fit_output.model
baseline_log_like = fit_output.log_like
source_names = list(model.sources.keys())
alt_models = _as_list(config.get('fitting.alternate_spatial_models'))
coord_range = config.get('fitting.extended_source_coord_range', 1.0)
logger.info(f"Sources in the model: {source_names}")
logger.info(f"Baseline log-likelihood: {baseline_log_like}")
logger.info(f"Alternate spatial models: {alt_models}")

In [ ]:
# # for sourcename in source_names:
# #     source = model.sources[sourcename]
# #     print(f"Checking source {source.name} for extension test")
# #     if source.name == 'URM':
# #         params = list(source.spatial_shape.parameters.items())
# #         # params = {k: v.value for k, v in params}
# #         print(f"Source {source.name} spatial shape: {params[0][1]}")
# #         params[0][1].free = False
# #         print(f"Source {source.name} spatial shape: {params[0][1].free}")

# for sourcename in source_names:
#     source = model.sources[sourcename]
#     print(f"Checking source {source.name} for extension test")
#     if source.name == 'URM':
#         params = list(source.spectrum.main._children.items())
#         spec_name, spec_func = params[0]
#         for pname, p in spec_func.parameters.items():
#             p.free = False
#         print(f"Source {source.name} spectrum: {spec_func}")
#         # params = {k: v.value for k, v in params}
#         # params[0][1].free = True
#         # print(f"Source {source.name} spatial shape: {params[0][1].free}")


In [ ]:
result = fit_output 
%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter

importlib.reload(source_fitter)
importlib.reload(model_generator)

alt_models = _as_list(config.get('fitting.alternate_spatial_models'))
if not alt_models:
    logger.info('No fitting.alternate_spatial_models configured; skipping extension test')

free_dbe = config.get('fitting.free_diffuse_norm', False)
threshold = config.get('likelihood_thresholds.extension_test', 16)
coord_range = config.get('fitting.extended_source_coord_range', 1.0)
runner = FitRunner(
    config_path=str(config.config_file),
    logger=logger,
    roi_template=config.get('roi.roi_template_path'),
)

model = fit_result.model
baseline_log_like = fit_result.log_like
source_names = list(model.sources.keys())
for source_name in source_names:
    if source_name == 'URM':
        logger.info(f'Skipping extension test for {source_name} (URM source)')
        continue
    other_sources = [n for n in model.sources.keys() if n != source_name]
    best_log_like = baseline_log_like
    best_model = model

    for alt_shape in alt_models:
        trial_model = model_generator.ModelGenerator.swap_spatial_shape(
            model, source_name, alt_shape, coord_range=coord_range, logger=logger,
        )
        model_generator.ModelGenerator.set_free(trial_model, other_sources, kind='spatial', free=False, free_diffuse=free_dbe, logger=logger)
        model_generator.ModelGenerator.set_free(trial_model, other_sources, kind='spectral', free=True, free_diffuse=free_dbe, logger=logger)

        step_name = f'Step2-{source_name}-Extension-{alt_shape}'
        step_dir = directory_manager.get_step_results_dir(step_name)
        model_file = model_generator.ModelGenerator.write_model_from_live(
            trial_model, str(directory_manager.get_model_file_path(step_name)), logger=logger,
        )
        trial_model.save("{1}/{0}.yml".format('curModel', step_dir), overwrite=True)
        model_generator.ModelGenerator.write_model_file_from_yaml("{1}/{0}.yml".format('curModel', step_dir), "{1}/{0}.model".format('curModel', step_dir), logger=logger)

        trial_result = runner.fit(
            model_file=str(model_file),
            step_dir=str(step_dir),
            compute_err=config.get('error_and_TS.error_extension', True),
            make_maps=False,
        )

        # logger.info(trial_model)
        # ModelGenerator.set_free(trial_model, other_sources, kind='spatial', free=False)
        # ModelGenerator.set_free(trial_model, other_sources, kind='spectral', free=False)

        # step_name = f'Step2-{source_name}-Extension-{alt_shape}'
        # step_dir = directory_manager.get_step_results_dir(step_name)
        # model_file = ModelGenerator.write_model_from_live(
        #     trial_model, str(directory_manager.get_model_file_path(step_name)), logger=logger,
        # )

In [ ]:
result = fit_output 

%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


if config.get('fitting.run_extension_test', True):
    result_ext = source_fitter.run_extension_test(result, config, logger, directory_manager)

In [ ]:
result_ext

In [ ]:
result = fit_output 

%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


if config.get('fitting.run_spectrum_test', True):
    result_ext = source_fitter.run_spectrum_test(result_ext, config, logger, directory_manager)

In [ ]:
result = fit_output 

%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


# if config.get('fitting.run_spectrum_test', True):
#     result_ext = source_fitter.run_spectrum_test(result_ext, config, logger, directory_manager)

if config.get('fitting.run_final_refit', True):
    result = source_fitter.run_final_refit(result, config, logger, directory_manager)

In [ ]:
result

In [ ]:
%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)

source_fitter.save_fit_summary(result, logger)

In [ ]:
from model_generator import ModelGenerator
num_sources = len(result.model.sources)
model_path = directory_manager.get_model_file_path('Final')
yml_path = str(model_path).replace('.model', '.yml') if str(model_path).endswith('.model') else f'{model_path}.yml'
result.model.save(yml_path, overwrite=True)
ModelGenerator.write_model_file_from_yaml(yml_path, str(model_path), logger=logger)